# Escalograma CWT — PETR4 (2020–2024)

Calcula e exibe o escalograma CWT com wavelet Morlet para o ativo PETR4, usando os retornos logarítmicos do preço de fechamento.

In [1]:
from pathlib import Path
OUTPUT_DIR = Path('/Users/fteodoro/Dropbox/Doutorado/Tese/figuras')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path('/Users/fteodoro/Dropbox/Doutorado/Fontes/LearnableWaveletLayer/data/PETR4.SA.csv')

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pywt
import csv
from datetime import datetime

plt.rcParams.update({'font.family': 'serif', 'font.size': 11})

## Leitura dos dados PETR4

Lê o CSV diretamente com o módulo `csv` (sem dependência de pandas).

In [3]:
dates_all = []
close_all = []

with open(DATA_PATH, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            d = datetime.strptime(row['Date'], '%Y-%m-%d')
            c = float(row['Close'])
            dates_all.append(d)
            close_all.append(c)
        except (ValueError, KeyError):
            pass

# Filter 2020-2024
start = datetime(2020, 1, 1)
end   = datetime(2024, 12, 31)
mask  = [start <= d <= end for d in dates_all]
dates = [d for d, m in zip(dates_all, mask) if m]
close = np.array([c for c, m in zip(close_all, mask) if m])

# Log-returns (drop first NaN)
log_ret = np.diff(np.log(close))
dates   = dates[1:]

print(f'PETR4 (2020–2024): {len(dates)} dias de negociação')

PETR4 (2020–2024): 1243 dias de negociação


## CWT com Wavelet Morlet

Aplica a Transformada Wavelet Contínua usando a wavelet de Morlet com escalas de 1 a 127.

In [4]:
scales = np.arange(1, 128)
coeffs, freqs = pywt.cwt(log_ret, scales, 'morl')
power = np.abs(coeffs) ** 2
print(f'Escalograma shape: {power.shape}  (escala × tempo)')

Escalograma shape: (127, 1243)  (escala × tempo)


## Visualização

Figura com dois painéis: série de preços (acima) e escalograma de potência (abaixo).

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

n_days = len(dates)
x_idx  = np.arange(n_days)

# Posições de tick no início de cada ano
year_ticks, year_labels, prev_year = [], [], None
for i, d in enumerate(dates):
    if d.year != prev_year:
        year_ticks.append(i)
        year_labels.append(str(d.year))
        prev_year = d.year

# -----------------------------------------------------------------------
# Correções de alinhamento:
#   1. sharex=True  → eixo x compartilhado (xlim e ticks sincronizados)
#   2. make_axes_locatable + eixo fantasma em ax_p → ambos os painéis
#      ficam com a mesma largura, eliminando o deslocamento causado pelo
#      colorbar que antes só existia no eixo inferior.
# -----------------------------------------------------------------------
fig, (ax_p, ax_s) = plt.subplots(
    2, 1, figsize=(13, 7),
    sharex=True,
    gridspec_kw={'height_ratios': [1, 3], 'hspace': 0.08}
)
fig.suptitle('Escalograma CWT — PETR4 (2020–2024), Wavelet Morlet',
             fontsize=13, fontweight='bold')

# --- Painel superior: série de preços ---
ax_p.plot(x_idx, close[1:], color='#4C72B0', lw=1.2)
ax_p.set_xlim(0, n_days - 1)
ax_p.set_ylabel('Preço (R$)', fontsize=10)
ax_p.set_title('PETR4 — Preço de Fechamento', fontsize=10)
ax_p.grid(alpha=0.3, ls=':')
ax_p.set_xticks(year_ticks)

# Eixo fantasma à direita de ax_p (mesmo tamanho do colorbar de ax_s)
div_p = make_axes_locatable(ax_p)
cax_p = div_p.append_axes("right", size="2%", pad=0.08)
cax_p.set_visible(False)

# --- Painel inferior: escalograma ---
x_edges = np.arange(-0.5, n_days + 0.5)
y_edges = np.arange(0.5, len(scales) + 1.5)

pcm = ax_s.pcolormesh(x_edges, y_edges, power, cmap='plasma', shading='flat')

ax_s.set_yscale('log')
ax_s.set_ylim(1, len(scales))

period_ticks = [1, 2, 4, 8, 16, 32, 64, 127]
ax_s.set_yticks(period_ticks)
ax_s.set_yticklabels([str(p) for p in period_ticks], fontsize=9)
ax_s.set_ylabel('Escala (dias de negociação)', fontsize=10)
ax_s.set_xlabel('Data', fontsize=10)
ax_s.set_xticks(year_ticks)
ax_s.set_xticklabels(year_labels, fontsize=9)

# Colorbar derivado de ax_s
div_s = make_axes_locatable(ax_s)
cax_s = div_s.append_axes("right", size="2%", pad=0.08)
cbar  = fig.colorbar(pcm, cax=cax_s)
cbar.set_label('Potência |CWT|²', fontsize=10)

out_path = OUTPUT_DIR / 'fig_scalogram_petr4.pdf'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Salvo em: {out_path}')
plt.show()
